[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/03_data_analysis_and_eda/01_types_of_data_and_storage.ipynb)

# Bài 01 — Làm việc với các loại dữ liệu và lưu trữ dữ liệu

> "Học Máy (Machine Learning) là một lĩnh vực nghiên cứu và xây dựng các giải thuật có khả năng học tự động từ dữ liệu để giải quyết các vấn đề cụ thể". Dữ liệu (data) đóng vai trò cực kỳ quan trọng và là yếu tố đầu tiên cần có khi thực hiện một ứng dụng Machine Learning.

### Mục tiêu bài học
- Tìm hiểu các kiểu dữ liệu cơ bản trong Machine Learning.
- Làm việc với dữ liệu **bảng**, **hình ảnh** và **văn bản** trên Python — và cách biến chúng thành **vector đặc trưng**.
- Các phương pháp lưu trữ dữ liệu: CSV, Parquet, Pickle, NumPy, HDF5 — ưu nhược điểm *(mở rộng)*.

### Nội dung
1. Dữ liệu có cấu trúc (bảng)
2. Dữ liệu không có cấu trúc: hình ảnh, văn bản, *(bổ sung)* chuỗi thời gian
3. Lưu trữ dữ liệu
4. Lấy dữ liệu từ nguồn bên ngoài: URL, GitHub, Google Drive, Dropbox, Kaggle *(bổ sung)*

> *So với bản gốc:* dữ liệu được lấy trực tiếp từ repo (link Dropbox gốc đã hỏng; cách tải từ URL/Drive/Dropbox/Kaggle trình bày ở mục 4); xử lý ảnh bằng Pillow + NumPy thay vì OpenCV; tách từ bằng biểu thức chính quy, có ví dụ tiếng Việt, NLTK là tuỳ chọn.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/03_data_analysis_and_eda

In [ ]:
import csv

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

set_style()

Trong Machine Learning, dữ liệu được chia thành 2 nhóm chính: **dữ liệu có cấu trúc** (Structured Data) và **dữ liệu không có cấu trúc** (Unstructured Data).

## 1. Dữ liệu có cấu trúc

- Dữ liệu thường biểu diễn ở dạng **bảng**.
- Mỗi hàng biểu diễn một **điểm dữ liệu (instance / sample)**.
- Các cột là các **đặc trưng (features)** và **nhãn (label)** của dữ liệu.
- Cách lưu trữ phổ biến nhất là file **.csv** (comma-separated values): mỗi dòng là một hàng, các giá trị phân cách bằng dấu phẩy (hoặc ký tự khác như tab, `;`).

Ví dụ: **Iris Dataset** — 150 bông hoa thuộc 3 loài, mỗi bông có 4 số đo.

<img src="https://imgur.com/SyhcVvY.jpg" width="450px"/>

In [ ]:
# Đọc file csv như file văn bản thông thường bằng thư viện csv của Python
with open("../../data/sample/iris.csv", encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile, delimiter=",")
    for row in list(reader)[::30]:
        print(row)

Để dễ dàng phân tích và xử lý dữ liệu dạng bảng, công cụ thường được sử dụng là **Pandas** (chương 01, bài 04):

In [ ]:
dataset = pd.read_csv("../../data/sample/iris.csv")
print(dataset.sample(5, random_state=0))
dataset.describe()

Biểu đồ **scatter matrix** (pairplot) cho thấy quan hệ giữa từng cặp feature, tô màu theo loài — một công cụ EDA rất hữu ích (bản gốc dùng `seaborn.pairplot`; ở đây dùng pandas để không cần thêm thư viện):

In [ ]:
colors = dataset["species"].map({"setosa": "#2563eb", "versicolor": "#dc2626", "virginica": "#16a34a"})
pd.plotting.scatter_matrix(dataset.iloc[:, :4], c=colors, figsize=(9, 9), diagonal="hist", alpha=0.7, s=15)
plt.suptitle("Iris — xanh: setosa, đỏ: versicolor, xanh lá: virginica")
plt.show()

Quan sát: `petal_length` và `petal_width` tách 3 loài rất rõ → đây là 2 feature quan trọng nhất. Đó chính là kiểu insight mà EDA (bài 02–03) giúp ta tìm ra **trước** khi train model.

Một số kỹ thuật tiền xử lý dữ liệu có cấu trúc (học kỹ ở chương 04):
- Xử lý dữ liệu bị thiếu (missing data)
- Chuẩn hoá dữ liệu (normalization / standardization)
- Rời rạc hoá dữ liệu (discretization / binning)
- Phát hiện và xử lý dữ liệu ngoại lai (outlier)
- Mã hoá biến phân loại (encoding)

Ví dụ **min-max scaling** đưa feature về đoạn $[0, 1]$ — hình dạng phân phối không đổi, chỉ thang đo thay đổi:

In [ ]:
from sklearn.preprocessing import minmax_scale

original_data = dataset["sepal_length"].to_numpy()
scaled_data = minmax_scale(original_data)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(original_data, bins=20, color="#ca8a04")
ax[0].set_title("Original data")
ax[1].hist(scaled_data, bins=20)
ax[1].set_title("Scaled data [0, 1]")
plt.show()

## 2. Dữ liệu không có cấu trúc

Các loại dữ liệu không có cấu trúc thường gặp: văn bản (text), hình ảnh (image), âm thanh (audio), video, chuỗi thời gian (time series), đồ thị (graph)...

Với dữ liệu không có cấu trúc, cần các phương pháp khác nhau để chuyển dữ liệu thô (raw data) thành **vector đặc trưng (feature vector)** trước khi áp dụng các giải thuật Machine Learning. Deep Learning (chương 27+) có khả năng **tự học** cách biểu diễn này.

### 2.1 Dữ liệu hình ảnh

#### a. Vector đặc trưng của ảnh
- Ảnh được lưu trong máy tính dưới dạng **ma trận số**, thể hiện màu sắc của các pixel.
- Ảnh RGB: mảng 3 chiều `(cao, rộng, 3)`, mỗi giá trị trong `[0, 255]` (kiểu `uint8`) là độ sáng của một kênh màu.

In [ ]:
from PIL import Image

image = np.asarray(Image.open("images/cat.jpg").convert("RGB"))
print("shape =", image.shape, "| dtype =", image.dtype)
print("Kênh Blue (5×5 pixel đầu):\n", image[:5, :5, 2])

plt.imshow(image)
plt.axis("off")
plt.show()

> 💡 Bản gốc dùng OpenCV (`cv2.imread`) — chú ý OpenCV đọc ảnh theo thứ tự kênh **BGR** chứ không phải RGB, nên khi hiển thị bằng matplotlib màu bị "ngược"; cần đảo kênh bằng `image[:, :, ::-1]`. Pillow đọc theo RGB nên không gặp vấn đề này.

Mỗi kênh màu là một ma trận:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for i, (ax, name, cmap) in enumerate(zip(axes, ["Red", "Green", "Blue"], ["Reds", "Greens", "Blues"])):
    ax.imshow(image[:, :, i], cmap=cmap)
    ax.set_title(f"Kênh {name}")
    ax.axis("off")
plt.show()

#### b. Một số kỹ thuật tiền xử lý ảnh

- **Thay đổi kích thước** (resize/crop): model thường yêu cầu ảnh đầu vào có kích thước cố định.

In [ ]:
resized_image = np.asarray(Image.fromarray(image).resize((25, 25)))
print("shape =", resized_image.shape)
plt.imshow(resized_image)
plt.title("Resize về 25×25")
plt.axis("off")
plt.show()

- **Chuyển sang ảnh xám** (grayscale) khi không quan tâm màu sắc: ma trận chỉ còn 2 chiều. Công thức chuẩn (ITU-R BT.601): $Y = 0.299R + 0.587G + 0.114B$.

In [ ]:
grayscale_image = image @ np.array([0.299, 0.587, 0.114])     # nhân ma trận theo chiều kênh màu
print("shape =", grayscale_image.shape)
plt.imshow(grayscale_image, cmap="gray")
plt.axis("off")
plt.show()

- **Làm giàu dữ liệu (data augmentation):** lật ảnh, xoay ảnh, dùng mặt nạ (mask), thay đổi độ sáng... để tạo thêm dữ liệu train (chương 33).

In [ ]:
pil = Image.fromarray(image)
augmented = {
    "Lật dọc": image[::-1],
    "Lật ngang": image[:, ::-1],
    "Lật cả hai": image[::-1, ::-1],
    "Xoay 20°": np.asarray(pil.rotate(20)),
    "Tăng sáng": np.clip(image.astype(int) + 60, 0, 255).astype(np.uint8),
}
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, img) in zip(axes, augmented.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.show()

In [ ]:
# "Mask" là một ma trận cho trước; nhân Hadamard (từng phần tử) để áp lên ảnh
mask = np.random.default_rng(0).random(grayscale_image.shape)
masked_image = mask * grayscale_image
plt.imshow(masked_image, cmap="gray")
plt.axis("off")
plt.show()

- **Duỗi ảnh thành vector:** ảnh `(H, W, 3)` → vector `H·W·3` chiều — cách đơn giản nhất để đưa ảnh vào model classical ML (chương 01, assignment 93: `image2vector`). Nhược điểm: mất thông tin về **vị trí không gian** của pixel → CNN (chương 29) khắc phục điều này.

In [ ]:
vector = resized_image.reshape(-1).astype(np.float32) / 255.0
print("vector đặc trưng:", vector.shape, "| min/max:", vector.min(), vector.max())

### 2.2 Dữ liệu văn bản

Mục đích của tiền xử lý văn bản là chuyển đổi các từ/ký tự thành các **vector** thể hiện đặc trưng của văn bản. Các bước thường gặp:

#### a. Tokenization — tách văn bản thành các từ (token)

In [ ]:
import re

sample_text = "I am writing a sample text."
print(sample_text.split())                          # tách theo khoảng trắng: dấu câu dính vào từ
print(re.findall(r"\w+|[^\w\s]", sample_text))      # tách cả dấu câu bằng regex

**Tiếng Việt khó hơn:** tiếng Việt là ngôn ngữ đơn lập — một **từ** có thể gồm nhiều **âm tiết** cách nhau bởi khoảng trắng ("máy tính", "học máy"). Tách theo khoảng trắng chỉ cho âm tiết. Cần công cụ tách từ (word segmentation) như `pyvi`, `underthesea`, `VnCoreNLP` — chương 32.

In [ ]:
vi_text = "Học máy là một lĩnh vực của trí tuệ nhân tạo."
syllables = re.findall(r"\w+", vi_text.lower())
print("Âm tiết:", syllables)
print("Từ (tách từ thủ công):", ["học_máy", "là", "một", "lĩnh_vực", "của", "trí_tuệ_nhân_tạo"])

#### b. Loại bỏ stopword

**Stopword** là các từ rất phổ biến nhưng ít mang ý nghĩa phân loại ("the", "is", "là", "của"...). Với bài toán phân loại văn bản, loại bỏ stopword giúp giảm kích thước từ vựng và hạn chế nhiễu.

⚠️ Không phải lúc nào cũng nên bỏ: "không" là stopword trong nhiều danh sách, nhưng "không tốt" ≠ "tốt" — bỏ đi sẽ làm sai bài toán phân tích cảm xúc!

In [ ]:
english_stopwords = {"i", "am", "a", "an", "the", "is", "it", "with", "of", "to", "and"}
tokens = re.findall(r"\w+", sample_text.lower())
print([t for t in tokens if t not in english_stopwords])

try:   # NLTK có danh sách stopword đầy đủ cho nhiều ngôn ngữ (tuỳ chọn: pip install nltk)
    from nltk.corpus import stopwords
    print(stopwords.words("english")[::10])
except (ImportError, LookupError):
    print("NLTK chưa được cài (hoặc chưa tải dữ liệu) — bỏ qua phần tuỳ chọn.")

#### c. Stemming / Lemmatization

Với tiếng Anh, từ biến đổi hình thức theo ngữ pháp (cats → cat, playing → play). **Stemming** cắt hậu tố theo quy tắc để đưa từ về dạng gốc (thô nhưng nhanh); **lemmatization** dùng từ điển (chính xác hơn). Tiếng Việt không biến hình nên không cần bước này.

In [ ]:
def naive_stem(word):
    """Stemmer đồ chơi: cắt vài hậu tố phổ biến (Porter stemmer phức tạp hơn nhiều)."""
    for suffix in ("ing", "ed", "es", "s"):
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[: -len(suffix)]
    return word


print([naive_stem(w) for w in ["cats", "playing", "played", "boxes", "is", "writing"]])

#### d. Xây dựng tập từ vựng (vocabulary)

Tập từ vựng thường là tập các từ xuất hiện trong dữ liệu **train** (sau khi tách từ, bỏ stopword...). Có thể bỏ các từ quá hiếm (nhiễu) hoặc quá phổ biến. Dùng một token đặc biệt **`<UNK>`** cho các từ không có trong từ vựng (gặp ở dữ liệu mới).

In [ ]:
sample_texts = ["I am playing with text", "It is a cat", "I like cat"]
tokenized = [re.findall(r"\w+", t.lower()) for t in sample_texts]
vocab = sorted(set(sum(tokenized, [])))

word2index = {"<UNK>": 0}
for word in vocab:
    word2index[word] = len(word2index)
index2word = {i: w for w, i in word2index.items()}
print(word2index)

encode = lambda text: [word2index.get(w, 0) for w in re.findall(r"\w+", text.lower())]
print("Mã hoá 'I like dogs':", encode("I like dogs"), "→ 'dogs' không có trong từ vựng → <UNK>")

#### e. Biểu diễn văn bản thành vector

**One-hot vector:** vector toàn 0, chỉ một vị trí bằng 1 — vị trí đó là chỉ số của từ. Dùng để biểu diễn từ, hoặc nhãn của bài toán phân loại.

**Bag-of-Words (BoW):** cộng one-hot của mọi từ trong câu → vector đếm số lần xuất hiện mỗi từ. Đơn giản, hiệu quả cho phân loại văn bản, nhưng **mất thứ tự từ** (chương 09, 32).

In [ ]:
V = len(word2index)


def one_hot(index, size=V):
    v = np.zeros(size)
    v[index] = 1.0
    return v


def bag_of_words(text):
    return sum(one_hot(i) for i in encode(text))


print("one-hot('cat')     :", one_hot(word2index["cat"]))
print("BoW('I like cat')  :", bag_of_words("I like cat"))
print("BoW('cat like I')  :", bag_of_words("cat like I"), "← giống hệt: BoW mất thứ tự từ!")

Biểu diễn hiện đại hơn — TF-IDF, word embedding (Word2Vec), Transformer — ở chương 32.

### 2.3 Dữ liệu chuỗi thời gian *(bổ sung)*

Dữ liệu được sắp theo **thời gian** — thứ tự là thông tin quan trọng. Biến thành feature bằng: giá trị trễ (lag), trung bình trượt, đặc trưng lịch (thứ, tháng, ngày lễ). Chú ý **không được xáo trộn ngẫu nhiên** khi chia train/test (chương 25).

In [ ]:
sales = pd.read_csv("../../data/sample/sales_daily.csv", parse_dates=["date"])
features = pd.DataFrame({
    "sales": sales["sales"],
    "lag_1": sales["sales"].shift(1),                 # doanh số hôm qua
    "lag_7": sales["sales"].shift(7),                 # cùng ngày tuần trước
    "rolling_mean_7": sales["sales"].shift(1).rolling(7).mean(),
    "dayofweek": sales["date"].dt.dayofweek,
})
features.iloc[7:12]

## 3. Lưu trữ dữ liệu

Tiền xử lý thường tốn nhiều thời gian và bộ nhớ. Vì vậy ta thường xử lý dữ liệu thô một lần, lưu lại kết quả (vector đặc trưng) ở định dạng phù hợp để dùng lại nhanh chóng.

| Định dạng | Ưu điểm | Nhược điểm | Dùng khi |
|---|---|---|---|
| **CSV** | đọc được bằng mắt, mọi công cụ hỗ trợ | chậm, file to, mất kiểu dữ liệu (ngày tháng, category) | trao đổi dữ liệu nhỏ |
| **Parquet** *(bổ sung)* | nén tốt, nhanh, giữ kiểu dữ liệu, đọc theo cột | cần `pyarrow` | dữ liệu bảng lớn — chuẩn trong data engineering |
| **Pickle** | lưu mọi object Python | ⚠️ **không an toàn** với file lạ, phụ thuộc phiên bản Python/thư viện | cache tạm của chính bạn |
| **NumPy `.npy/.npz`** | nhanh, đơn giản cho mảng số | chỉ cho mảng | feature matrix, embedding |
| **HDF5** | dữ liệu số rất lớn, cấu trúc phân cấp, đọc từng phần | cần `h5py`/`tables` | ảnh, tín hiệu khoa học |
| **joblib** | lưu model scikit-learn hiệu quả | giống pickle về bảo mật | lưu model (chương 39) |

> ⚠️ **Bảo mật:** `pickle.load` (và `joblib.load`, `torch.load` với file cũ) có thể **thực thi mã tuỳ ý** chứa trong file. Không bao giờ load pickle từ nguồn không tin cậy.

In [ ]:
import os
import pickle
import tempfile
import time
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
df = pd.read_csv("../../data/sample/fraud_transactions.csv")


def timed(label, fn):
    start = time.perf_counter()
    fn()
    return (time.perf_counter() - start) * 1000


rows = []
rows.append(("CSV", timed("w", lambda: df.to_csv(tmp / "d.csv", index=False)),
             timed("r", lambda: pd.read_csv(tmp / "d.csv")), os.path.getsize(tmp / "d.csv")))
rows.append(("Pickle", timed("w", lambda: df.to_pickle(tmp / "d.pkl")),
             timed("r", lambda: pd.read_pickle(tmp / "d.pkl")), os.path.getsize(tmp / "d.pkl")))
try:
    rows.append(("Parquet", timed("w", lambda: df.to_parquet(tmp / "d.parquet")),
                 timed("r", lambda: pd.read_parquet(tmp / "d.parquet")), os.path.getsize(tmp / "d.parquet")))
except ImportError:
    print("Chưa cài pyarrow → bỏ qua Parquet (pip install pyarrow)")

pd.DataFrame(rows, columns=["định dạng", "ghi (ms)", "đọc (ms)", "kích thước (byte)"]).set_index("định dạng")

Lưu mảng NumPy (vd: ma trận feature của ảnh) bằng `.npz` (nhiều mảng trong một file, có nén):

In [ ]:
X_img = np.random.default_rng(0).random((500, 64)).astype(np.float32)
y_img = np.arange(500) % 10
np.savez_compressed(tmp / "features.npz", X=X_img, y=y_img)
loaded = np.load(tmp / "features.npz")
print(loaded["X"].shape, loaded["y"][:10], "| file:", os.path.getsize(tmp / "features.npz"), "byte")

with open(tmp / "vocab.pkl", "wb") as f:      # pickle cho object Python (vd: từ điển word2index)
    pickle.dump(word2index, f)
with open(tmp / "vocab.pkl", "rb") as f:
    print(pickle.load(f) == word2index)

HDF5 (tuỳ chọn — cần `pip install h5py`):

```python
import h5py
with h5py.File("images.h5", "w") as f:
    f.create_dataset("X", data=X_img, compression="gzip")
with h5py.File("images.h5", "r") as f:
    batch = f["X"][:32]          # chỉ đọc 32 hàng đầu — không cần nạp cả file vào RAM
```

## 4. Lấy dữ liệu từ nguồn bên ngoài *(bổ sung — tương ứng mục 0.1, 0.2 của bản gốc)*

Dữ liệu của khoá học nằm sẵn trong repo, nhưng dự án thật thường bắt đầu bằng việc **tải dữ liệu về**. Các nguồn hay gặp:

| Nguồn | Cách lấy |
|---|---|
| URL trực tiếp (http/https) | `pd.read_csv("https://…/data.csv")` — pandas đọc thẳng từ URL, kể cả file `.gz`/`.zip` |
| GitHub | dùng link **raw**: `https://raw.githubusercontent.com/<user>/<repo>/<branch>/<path>` |
| Google Drive | trên Colab: `from google.colab import drive; drive.mount("/content/drive")` ([bài 00 · 03](../00_course_orientation/03_google_colab_tips.ipynb)); file chia sẻ công khai: `pip install gdown` rồi `gdown <file_id>` |
| Dropbox | trong link chia sẻ, đổi `?dl=0` thành `?dl=1` để tải thẳng file thay vì mở trang xem trước |
| Kaggle | `pip install kaggle`, đặt token API vào `~/.kaggle/kaggle.json`, rồi `kaggle datasets download -d <owner>/<dataset>` — đọc kỹ điều khoản sử dụng dữ liệu |

```python
# Ví dụ (cần mạng — không chạy trong bài):
df = pd.read_csv("https://raw.githubusercontent.com/<user>/<repo>/main/data.csv")
```

**Nguyên tắc tải dữ liệu tái lập được:**
1. Tải **một lần** vào `data/raw/` (thư mục này không được commit — xem `.gitignore`), **không sửa tay** file gốc.
2. Ghi lại **nguồn, ngày tải và checksum** (MD5/SHA-256) — để biết chắc người khác (hoặc chính bạn sau 6 tháng) đang dùng *đúng* file đó.
3. Code xử lý đọc từ `data/raw/`, ghi kết quả vào `data/processed/`.

Module `ml_course.real_datasets` của khoá học làm đúng như vậy cho hai bộ ảnh thật dùng ở các chương sau (Fashion-MNIST ~30 MB, CIFAR-10 ~170 MB): tải lần đầu, **kiểm tra MD5** với giá trị công bố, lưu cache, lần sau đọc từ đĩa:

```python
from ml_course.real_datasets import load_fashion_mnist
X_train, y_train, X_test, y_test = load_fashion_mnist()   # lần đầu tải ~30 MB vào data/raw/fashion-mnist
```

Checksum là "dấu vân tay" của file: đổi dù chỉ một byte, checksum đổi hoàn toàn.

In [ ]:
import hashlib


def file_md5(path, chunk_size=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()


original = Path("../../data/sample/iris.csv").read_bytes()
copy_path = tmp / "iris_copy.csv"
copy_path.write_bytes(original)
print("MD5 bản gốc        :", file_md5("../../data/sample/iris.csv"))
print("MD5 bản sao        :", file_md5(copy_path))
copy_path.write_bytes(original.replace(b"5.1", b"5.2", 1))      # sửa đúng MỘT ký tự
print("MD5 sau khi sửa 1 ký tự:", file_md5(copy_path))

## 🧠 Kiểm tra nhanh

1. Ảnh RGB 64×48 duỗi thành vector có bao nhiêu chiều?
2. Vì sao "không" có thể là stopword nguy hiểm với bài toán phân tích cảm xúc?
3. Câu "chó cắn người" và "người cắn chó" có biểu diễn Bag-of-Words giống nhau không?
4. Bạn nhận một file `.pkl` từ một người lạ trên mạng. Có nên `pickle.load` nó không?
5. Vì sao nên ghi lại checksum của file dữ liệu tải về, và vì sao không commit `data/raw/` lên Git?

## 📌 Tổng kết

- Dữ liệu có cấu trúc (bảng): mỗi hàng là một mẫu, mỗi cột là một feature/nhãn — dùng pandas.
- Dữ liệu không có cấu trúc (ảnh, văn bản, chuỗi thời gian...) cần được biến thành vector đặc trưng.
- Ảnh = mảng số `(H, W, C)`; tiền xử lý: resize, grayscale, augmentation, chuẩn hoá về [0, 1].
- Văn bản: tokenization → stopword → stemming → vocabulary (`<UNK>`) → one-hot / Bag-of-Words.
- Lưu trữ: CSV để trao đổi, Parquet cho bảng lớn, `.npz` cho mảng, cẩn thận với pickle.
- Dữ liệu tải từ ngoài: tải một lần vào `data/raw/`, ghi nguồn + checksum, không sửa tay file gốc.

---
<sub>Bài gốc: *Tutorial 3: Làm việc với dữ liệu* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), thư mục Archives (MIT License). Đã cập nhật: bỏ phụ thuộc Drive/Dropbox/OpenCV/seaborn/NLTK, thêm tiếng Việt, Bag-of-Words, chuỗi thời gian, Parquet, cảnh báo bảo mật pickle, so sánh định dạng.</sub>